# ifcsteel LoRA eğitimi (Kaggle, ücretsiz GPU)

Mac'te 16 GB yetmediği için LoRA burada eğitilir; çıkan `.safetensors` dosyası Mac'te `gece.command` ile kullanılır.

1. Sağ panel: **Session options → Accelerator: GPU T4 x2**, **Internet: On** (telefon doğrulaması ister).
2. **Add Input → Upload → `lora_data.zip`** (Mac'te `lora_data` klasörünü sıkıştır; içinde 0001.png, 0001.txt ... olmalı).
3. Sağ üstte **Save Version → Save & Run All (Commit)**. Tarayıcıyı kapatabilirsin; en fazla 12 saat çalışır.
4. Bitince versiyonun **Output** sekmesinden `ifcsteel_lora_sonuc.zip`'i indir.


In [ ]:
!nvidia-smi
import glob, os
hits = sorted(glob.glob('/kaggle/input/**/0001.png', recursive=True))
assert hits, "Veri seti bulunamadı: Add Input ile lora_data.zip'i yükle"
import shutil
# Kaggle inputs are read-only and ai-toolkit caches latents next to the images: work on a copy
DATA = '/tmp/lora_data'
shutil.rmtree(DATA, ignore_errors=True)
shutil.copytree(os.path.dirname(hits[0]), DATA)
print(DATA, len(glob.glob(DATA + '/[0-9][0-9][0-9][0-9].png')), 'görsel')


In [ ]:
!git clone -q --depth 1 https://github.com/ostris/ai-toolkit /tmp/ai-toolkit
!pip install -q torch==2.13.0 torchvision==0.28.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu128 || pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!cd /tmp/ai-toolkit && pip install -q -r requirements.txt


In [ ]:
import os, sys
os.environ['HF_HOME'] = '/tmp/hf'                 # model (~33 GB) stays out of the saved output
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
sys.path.insert(0, '/tmp/ai-toolkit')
from toolkit.job import run_job

STEPS = 2000          # 71 görsel için ~28 tur; her 250 adımda kayıt ve önizleme
PREVIEWS = ['a lighthouse on rocks with crashing waves',
            'a fox curled around a crescent moon with pine branches']

job = {
  'job': 'extension',
  'config': {
    'name': 'ifcsteel',
    'process': [{
      'type': 'sd_trainer',
      'training_folder': '/kaggle/working/output',
      'device': 'cuda:0',
      'trigger_word': 'ifcsteel',
      'network': {'type': 'lora', 'linear': 16, 'linear_alpha': 16},
      'save': {'dtype': 'float16', 'save_every': 250, 'max_step_saves_to_keep': 8},
      'datasets': [{'folder_path': DATA, 'caption_ext': 'txt', 'caption_dropout_rate': 0.05,
                    'cache_latents_to_disk': True, 'resolution': [512, 768]}],
      'train': {'batch_size': 1, 'steps': STEPS, 'gradient_accumulation_steps': 1,
                'train_unet': True, 'train_text_encoder': False, 'gradient_checkpointing': True,
                'noise_scheduler': 'flowmatch', 'optimizer': 'adamw8bit', 'lr': 1e-4, 'dtype': 'bf16'},
      'model': {'name_or_path': 'Tongyi-MAI/Z-Image-Turbo', 'arch': 'zimage',
                # Turbo is step-distilled: train through ostris' adapter or the distillation breaks
                'assistant_lora_path': 'ostris/zimage_turbo_training_adapter/zimage_turbo_training_adapter_v2.safetensors',
                'quantize': True, 'quantize_te': True, 'low_vram': True},
      'sample': {'sampler': 'flowmatch', 'sample_every': 250, 'width': 768, 'height': 768,
                 'prompts': [f'ifcsteel style, {p}, black laser-cut steel cut file, one connected piece' for p in PREVIEWS],
                 'neg': '', 'seed': 42, 'walk_seed': False, 'guidance_scale': 1.0, 'sample_steps': 8},
    }],
  },
  'meta': {'name': 'ifcsteel', 'version': '1.0'},
}
run_job(job)


In [ ]:
!cd /kaggle/working/output && zip -q -r /kaggle/working/ifcsteel_lora_sonuc.zip ifcsteel -x '*.pt' && ls -lh /kaggle/working/ifcsteel_lora_sonuc.zip
